#  3B1B 线代核心笔记 (Ep.9 - Ep.12)

## P9: 点积与对偶性 (Dot Products & Duality)
* **代数计算**：对应分量相乘后相加，结果为标量。 $v \cdot w = v_x w_x + v_y w_y$
* **几何直觉 (投影)**：将向量 $w$ 正交投影到 $v$ 上，投影长度乘 $v$ 的长度。方向相同为正，垂直为 $0$，相反为负。
* **核心思想 (对偶性)**：多维空间到一维数轴的线性变换，在数学上等价于与某个特定向量的**点积**运算。
* **ML联想**：线性回归 $y = w^T x + b$，本质就是将高维特征 $x$ 投影到一维预测轴上，权重 $w$ 就是投影的参考向量。

---

## P10-P11: 叉积 (Cross Products)
* **二维叉积**：两向量围成的**平行四边形面积**，带正负号（即行列式的值）。
* **三维叉积**：结果是一个全新的向量。
  * **大小 (长度)**：底面平行四边形的面积。
  * **方向**：垂直于原向量构成的平面，严格符合**右手定则**。
* **线性变换视角**：“输入任意向量 $\rightarrow$ 输出平行六面体体积” 这个线性变换，根据对偶性，必然对应一个向量的点积，这个“被迫存在”的向量就是叉积。

---

## P12: 基变换 (Change of Basis)
* **核心概念**：不同坐标系对同一个向量的描述不同，因为他们使用的**基向量 (Basis Vectors)** 不同。
* **坐标转换矩阵 $P$**：将“对方的基向量”写成列向量拼成的矩阵。
  * **对方 $\rightarrow$ 我们**：$\text{我们的坐标} = P \times \text{对方的坐标}$
  * **我们 $\rightarrow$ 对方**：$\text{对方的坐标} = P^{-1} \times \text{我们的坐标}$
* **相似矩阵公式 $A = P^{-1}MP$**：
  * 如果想在对方的坐标系下，执行我们坐标系里的线性变换 $M$：
  * 先把坐标转过来 ($P$) $\rightarrow$ 执行变换 ($M$) $\rightarrow$ 再把结果转回去 ($P^{-1}$)。
* **DL联想**：神经网络隐藏层的矩阵乘法，本质上是在不断改变数据的“基”（特征空间），把复杂难分的数据扭曲、变换到更容易被切分的新坐标系中。

# 李宏毅机器学习核心笔记：模型误差与正则化

## 1. 误差从何而来？(Bias vs. Variance)
打靶比喻：模型的预测值就像打靶，真实值是靶心。
* **偏差 (Bias)**：瞄准的方向偏了。模型太简单（如用直线拟合二次曲线），连训练集都学不好。
  * **症状**：**Underfitting（欠拟合）**，Training Loss 下不去。
* **方差 (Variance)**：打得很散。模型太复杂，过度放大了数据中的噪声。
  * **症状**：**Overfitting（过拟合）**，Training Loss 很低，但 Testing/Validation Loss 很高。
* **权衡 (Trade-off)**：模型越复杂，Bias 越小，但 Variance 越大。我们需要找到使得 Total Error 最小的平衡点。

---

## 2. 疑难杂症诊断与处方 (Diagnosis & Solutions)

### 症状 A：Training Loss 很大 (欠拟合 / High Bias)
* **诊断**：模型“脑容量”不够，或者大海捞针没捞到。
* **处方**：
  1. **增加特征 (More Features)**：引入高次项或其他维度特征。
  2. **换更复杂的模型**：比如加深神经网络层数。

### 症状 B：Train Loss 小，Test Loss 大 (过拟合 / High Variance)
* **诊断**：模型在“死记硬背”，泛化能力差。
* **处方**：
  1. **收集更多数据**：（最有效，但成本高）。
  2. **数据增强 (Data Augmentation)**：对现有数据进行平移、翻转、加噪等。
  3. **限制模型能力**：减少参数量、Early Stopping（早停法）。
  4. **正则化 (Regularization)**（见下一节）。

---

## 3. 正则化 (Regularization)
* **核心思想**：在原有的 Loss 函数后面，加上一项对权重 $w$ 的惩罚项，强迫模型选择更小、更接近 $0$ 的参数。
* **公式 (L2正则化 / Weight Decay)**：$L = \text{Original Loss} + \lambda \sum w_i^2$
* **为什么有效？**：
  * 参数越小，说明模型对输入 $x$ 的扰动越**不敏感**。
  * 曲线越平滑 (Smoother)，抗噪声能力越强，泛化能力越好。
* **注意**：正则化通常**不惩罚偏置 $b$**。因为 $b$ 只控制曲线的上下平移，不影响曲线的平滑度。

---

## 4. 模型评估与选择 (Model Selection)
* **铁律**：**绝对不能用 Testing Set 的结果来调参选模型！**（这会导致模型在测试集上产生过拟合）。
* **正确做法**：将训练数据再切分，划分为 **Training Set** 和 **Validation Set（验证集）**。
  * 用 Training Set 更新参数。
  * 用 Validation Set 挑选最好的模型架构或超参数（如正则化系数 $\lambda$）。
* **进阶做法**：**N折交叉验证 (N-fold Cross Validation)**。将数据切成 N 份，轮流做验证集求平均 Loss，能让模型评估更稳定。

In [10]:
import numpy as np
import torch
from torch.utils import data
from d2l import torch as d2l

# 生成数据集
true_w = torch.tensor([2, -3.4])
true_b = 4.2
features, labels = d2l.synthetic_data(true_w, true_b, 1000)

In [11]:
# 读取数据集
def load_array(data_arrays, batch_size, is_train=True):
    """ 构建一个 Pytorch 数据迭代器 """

    # [重点 1: 打包] 
    # data_arrays 是个元组 (features, labels)。
    # *data_arrays 解包后，TensorDataset 像拉链一样把特征和标签按行一一对应绑死。
    dataset = data.TensorDataset(*data_arrays)

    # [重点 2: 发牌]
    # DataLoader 接收打包好的 dataset。
    # batch_size: 决定每次抓几条数据（这里是 10 条）。
    # shuffle=is_train: 训练时必须洗牌(True)，防止模型背数据顺序；测试时不用(False)。
    return data.DataLoader(dataset, batch_size, shuffle=is_train)

# 设定每批次抓取 10 条数据
batch_size = 10

# 实例化发牌机，得到迭代器 data_iter
# 后面训练时，我们就可以用 `for X, y in data_iter:` 优雅地拿数据了
data_iter = load_array((features, labels), batch_size)

next(iter(data_iter))

[tensor([[-0.1588, -1.3594],
         [-0.6812, -0.7681],
         [-0.5327, -0.6354],
         [-0.1273,  2.7120],
         [-0.2591, -0.3075],
         [-1.4182,  1.2689],
         [ 1.5555, -1.7404],
         [-1.1012, -0.6836],
         [ 0.8977, -0.8380],
         [-0.0179, -1.0521]]),
 tensor([[ 8.5048],
         [ 5.4489],
         [ 5.2856],
         [-5.2774],
         [ 4.7379],
         [-2.9438],
         [13.2224],
         [ 4.3200],
         [ 8.8401],
         [ 7.7494]])]

In [12]:
# 定义模型
# nn是神经网络 (Neural Network) 的缩写，PyTorch 的神级模块
from torch import nn

# [重点 1: nn.Linear(2, 1) - 核心计算层]
# 2 代表输入特征数 (in_features)，因为我们之前造的数据有 2 个特征。
# 1 代表输出特征数 (out_features)，因为我们只需要预测 1 个连续值（比如房价）。
# 黑盒魔法：只要写下这句，PyTorch 就已经在底层偷偷帮你建好了形状匹配的权重 w 和偏置 b！

# [重点 2: nn.Sequential - 容器]
# 它就像一块“乐高底板”或者一根“管道”。
# 把 nn.Linear 塞进去后，以后数据扔进 net，就会自动顺着管道经过里面的每一层。
net = nn.Sequential(nn.Linear(2, 1))

In [13]:
# 初始化模型参数

# [重点 1: net[0] 是什么？]
# net 是我们刚才建的 nn.Sequential 容器。
# net[0] 就是这个管道里的第一层，也就是那个 nn.Linear(2, 1) 网络层。

# [重点 2: weight 和 bias]
# 它们就是这一层的权重 w 和 偏置 b。

# [重点 3: .data 与 下划线魔法 _ ]
# .data 是直接操作底层的数据张量。
# normal_(0, 0.01) : 从均值为 0、标准差为 0.01 的正态分布中随机抽样来替换掉原来的 w。
# fill_(0) : 直接把 b 强行赋值为 0。
# ⚠️ 注意这里的方法名后面都带了一个下划线 `_` ！
# 在 PyTorch 中，这代表“就地修改 (In-place operation)”，也就是不创建新变量，直接把原内存里的值覆盖掉，极其省内存。
net[0].weight.data.normal_(0, 0.01)
net[0].bias.data.fill_(0)

tensor([0.])

In [14]:
# 定义损失函数

# [重点 1: MSE 是什么？]
# MSE = Mean Squared Error (均方误差)。
# 它的底层逻辑就是昨天你自己手写的数学公式：将预测值和真实值相减，求平方，再对整个 Batch 求平均。
# 公式：L = 1/n * Σ(y_hat - y)^2

# [重点 2: 这是一个“计算器”对象]
# 注意这里带有括号 ()，这意味着我们创建（实例化）了一个“误差计算器”。
# 待会儿在训练循环里，我们只需要把预测值和真实值扔给它：`l = loss(net(X), y)`，它就会吐出一个误差分值。
loss = nn.MSELoss()

In [15]:
# 定义优化器

# [重点 1: optim 模块]
# torch.optim 包含了深度学习里所有的优化算法，SGD（随机梯度下降）是最经典的一个。

# [重点 2: net.parameters()]
# 告诉优化器，你要去打理哪些参数。
# net.parameters() 会自动把我们在第 3 步里 nn.Linear 底层生成的 w 和 b 全部打包交给 SGD。

# [重点 3: lr (Learning Rate, 学习率)]
# 它是深度学习中最重要的“超参数”之一。决定了每次沿着梯度下山的“步伐有多大”。
# 这里设为 0.03，是一个经典且相对安全的经验值。
trainer = torch.optim.SGD(net.parameters(), lr=0.03)

In [16]:
# 循环

num_epoch = 3 # 设定把整个数据集看几遍（这里是 3 遍）

# [外层循环]：控制总共训练几轮 (Epoch)
for epoch in range(num_epoch):

    # [内层循环]：控制每一次从发牌机里抽几张牌 (Batch)
    # data_iter 每次吐出 10个特征(X) 和 10个真实标签(y)
    for X, y in data_iter:

        # 动作 1：前向传播与算误差 (Forward & Loss)
        # net(X) 就是让模型去猜这 10 个房价
        # loss(猜的值, 真实值) 算出这 10 个样本的平均均方误差 l
        l = loss(net(X), y)

        # 动作 2：清空前人遗留的梯度 (Zero Gradients) ⚠️极其重要！
        trainer.zero_grad()

        # 动作 3：反向传播 (Backward)
        # 顺着误差 l，利用微积分链式法则，往回算出每个参数 (w, b) 需要修改的方向和力度 (梯度)
        l.backward()

        # 动作 4：更新参数 (Step)
        # 打工人 (SGD 优化器) 按照刚才算出的梯度，和预设的学习率 (0.03)，把 w 和 b 往正确的方向拨动一点点
        trainer.step()

    # [一轮结束后的检阅]
    # 把上帝视角的整个 1000 条特征 (features) 全丢进去算一次总误差
    # 打印出来看看 Loss 是不是在稳步下降
    l = loss(net(features), labels)
    print(f'epoch {epoch + 1}, loss {l:f}')

w = net[0].weight.data
print('w的估计误差：', true_w - w.reshape(true_w.shape))
b = net[0].bias.data
print('b的估计误差：', true_b - b)

epoch 1, loss 0.000352
epoch 2, loss 0.000106
epoch 3, loss 0.000105
w的估计误差： tensor([-0.0001,  0.0004])
b的估计误差： tensor([-9.6798e-05])
